# Experiment 3: Candidate-Elimination Algorithm

## Overview
The Candidate-Elimination algorithm maintains the Version Space - the set of all hypotheses consistent with the training examples. It tracks two boundaries:
- **General boundary (G)**: Most general hypotheses consistent with data
- **Specific boundary (S)**: Most specific hypotheses consistent with data

## Algorithm Steps
1. Initialize G to set of most general hypotheses
2. Initialize S to set of most specific hypotheses
3. For each training example:
   - If positive: remove from G any hypothesis that doesn't cover it, specialize S if needed
   - If negative: remove from S any hypothesis that covers it, generalize G if needed
4. Return all hypotheses in the version space

In [ ]:
import pandas as pd
import numpy as np
from itertools import product

print("Candidate-Elimination Algorithm Implementation")
print("=" * 60)

## Sample Dataset

In [ ]:
# Create sample dataset
data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'Temp': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak'],
    'Water': ['Warm', 'Cool', 'Warm', 'Cool'],
    'Forecast': ['Same', 'Same', 'Change', 'Change'],
    'EnjoySports': ['Yes', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(data)
print("\nTraining Dataset:")
print(df)

## Candidate-Elimination Implementation

In [ ]:
def candidate_elimination(dataset, target_column):
    """
    Implement Candidate-Elimination algorithm
    
    Parameters:
    -----------
    dataset : pd.DataFrame
        Training dataset
    target_column : str
        Name of the target column
    
    Returns:
    --------
    S : list
        Specific boundary (most specific hypotheses)
    G : list
        General boundary (most general hypotheses)
    """
    attributes = [col for col in dataset.columns if col != target_column]
    n_attrs = len(attributes)
    
    # Get unique values for each attribute
    attr_values = {}
    for attr in attributes:
        attr_values[attr] = list(dataset[attr].unique())
    
    print("\n" + "="*60)
    print("CANDIDATE-ELIMINATION ALGORITHM")
    print("="*60)
    
    # Initialize S to the most specific hypothesis
    S = [['0'] * n_attrs]  # 0 represents no example seen yet
    
    # Initialize G to the most general hypothesis
    G = [['?'] * n_attrs]
    
    print(f"\nInitial State:")
    print(f"S (Specific): {S}")
    print(f"G (General): {G}")
    
    # Process each training example
    for idx, (example_idx, example) in enumerate(dataset.iterrows(), 1):
        x = list(example[attributes].values)
        label = example[target_column]
        
        print(f"\n--- Example {idx}: {dict(example[attributes])} | Target: {label} ---")
        
        if label == 'Yes':
            print("Positive Example:")
            
            # Remove from G any hypothesis that doesn't cover this example
            G_new = []
            for g in G:
                if matches_hypothesis(x, g, attributes):
                    G_new.append(g)
                else:
                    print(f"  Removed from G: {g}")
            G = G_new
            
            # Specialize S if needed
            S_new = []
            for s in S:
                if matches_hypothesis(x, s, attributes):
                    S_new.append(s)
                else:
                    # Specialize s to cover x
                    specialized = list(s)
                    for attr_idx, attr in enumerate(attributes):
                        if specialized[attr_idx] == '0' or specialized[attr_idx] == '?':
                            specialized[attr_idx] = x[attr_idx]
                    S_new.append(specialized)
                    print(f"  Specialized S: {specialized}")
            S = S_new
        
        else:  # label == 'No'
            print("Negative Example:")
            
            # Remove from S any hypothesis that covers this example
            S_new = []
            for s in S:
                if not matches_hypothesis(x, s, attributes):
                    S_new.append(s)
                else:
                    print(f"  Removed from S: {s}")
            S = S_new
            
            # Generalize G if needed
            G_new = []
            for g in G:
                if not matches_hypothesis(x, g, attributes):
                    G_new.append(g)
                else:
                    # Generalize g to exclude x
                    generalized = list(g)
                    for attr_idx, attr in enumerate(attributes):
                        if generalized[attr_idx] != '?':
                            generalized[attr_idx] = '?'
                    G_new.append(generalized)
                    print(f"  Generalized G: {generalized}")
            G = G_new
        
        print(f"After processing:")
        print(f"  S: {S}")
        print(f"  G: {G}")
    
    return S, G, attributes

def matches_hypothesis(example, hypothesis, attributes):
    """
    Check if example matches hypothesis
    """
    for i, (attr, hyp_val) in enumerate(zip(attributes, hypothesis)):
        if hyp_val == '0':
            continue
        if hyp_val != '?' and hyp_val != example[i]:
            return False
    return True

# Run algorithm
S, G, attributes = candidate_elimination(df, 'EnjoySports')

## Results

In [ ]:
print("\n" + "="*60)
print("FINAL VERSION SPACE")
print("="*60)

print(f"\nSpecific Boundary (S):")
for i, s in enumerate(S, 1):
    print(f"  {i}. {dict(zip(attributes, s))}")

print(f"\nGeneral Boundary (G):")
for i, g in enumerate(G, 1):
    print(f"  {i}. {dict(zip(attributes, g))}")

print(f"\nVersion Space Size: {len(S) + len(G)} hypotheses")
print(f"Specific Hypotheses: {len(S)}, General Hypotheses: {len(G)}")

## Viva Questions

### Basic Concepts
1. **What is the Version Space in the Candidate-Elimination algorithm?**
   - Set of all hypotheses consistent with the training examples
   - Bounded by specific and general boundaries

2. **Define the Specific boundary (S) and General boundary (G).**
   - S: Most specific hypotheses consistent with positive examples
   - G: Most general hypotheses consistent with data

3. **How does Candidate-Elimination differ from FIND-S?**
   - FIND-S returns one hypothesis
   - CE returns all consistent hypotheses (version space)
   - CE maintains both S and G boundaries

### Algorithm Questions
4. **Describe how Candidate-Elimination handles positive examples.**
   - Remove from G hypotheses that don't cover the example
   - Keep in S hypotheses that cover it, specialize others

5. **How does the algorithm handle negative examples?**
   - Remove from S hypotheses that cover the example
   - Generalize G hypotheses that cover it to exclude it

6. **What happens when the version space becomes empty?**
   - Indicates no consistent hypothesis exists
   - Suggests data is inconsistent or has errors

### Practical Questions
7. **What is the advantage of maintaining the entire version space?**
   - Can make probabilistic predictions
   - Can identify which examples are most informative
   - More robust than single hypothesis approach

8. **How can the version space be used for prediction on new examples?**
   - If all hypotheses in VS classify as positive: classify positive
   - If all classify as negative: classify negative
   - If mixed: uncertain prediction

9. **What are limitations of Candidate-Elimination?**
   - Version space can be exponentially large
   - Cannot handle continuous attributes easily
   - Assumes consistent and complete data

10. **How is this algorithm useful in practice?**
    - Helps understand concept learning
    - Can identify most informative examples (active learning)
    - Educational tool for understanding hypothesis spaces